# Import

In [260]:
import pandas as pd
import numpy as np

# Constantes

In [261]:
# Processed Data Path
PATH_PROCESSED = "../data/processed/"

# Palpites
FILE_TIPS = "palpites_fg_processados.csv"
# Paises
FILE_PAISES = "apoio_paises.csv"

# ETL

## Leitura e Join com apoio

In [262]:
df_paises = pd.read_csv(PATH_PROCESSED + FILE_PAISES)

In [263]:
df_tips_table = pd.read_csv(PATH_PROCESSED + FILE_TIPS)
df_tips_table["nm_pais"] = df_tips_table["nm_time_casa"]

In [264]:
df_tips_table_2 = pd.merge(df_tips_table, df_paises, on='nm_pais', how='left')
df_tips_table_2.drop(["nm_pais"], axis=1, inplace=True)

In [265]:
df_tips_table_3 = df_tips_table_2.dropna()
df_tips_table_3['id_pais'] = df_tips_table_3['id_pais'].astype(int)

## Gerar classificação

In [266]:
df = df_tips_table_3.copy()

# resultados por mandante
home = df.rename(columns={
    "nm_time_casa": "team",
    "nm_time_fora": "opp",
    "vl_time_casa": "gf",
    "vl_time_fora": "ga",
})

In [267]:
home["pts"] = np.select(
    [home["gf"] > home["ga"], home["gf"] == home["ga"]],
    [3, 1],
    default=0,
)

home["v"] = (home["gf"] > home["ga"]).astype(int)
home["e"] = (home["gf"] == home["ga"]).astype(int)
home["d"] = (home["gf"] < home["ga"]).astype(int)

In [268]:
home["v"] = (home["gf"] > home["ga"]).astype(int)
home["e"] = (home["gf"] == home["ga"]).astype(int)
home["d"] = (home["gf"] < home["ga"]).astype(int)

In [269]:
# resultados por visitante (espelha o jogo)
away = df.rename(columns={
    "nm_time_fora": "team",
    "nm_time_casa": "opp",
    "vl_time_fora": "gf",
    "vl_time_casa": "ga",
})

In [270]:
away["pts"] = np.select(
    [away["gf"] > away["ga"], away["gf"] == away["ga"]],
    [3, 1],
    default=0,
)

away["v"] = (away["gf"] > away["ga"]).astype(int)
away["e"] = (away["gf"] == away["ga"]).astype(int)
away["d"] = (away["gf"] < away["ga"]).astype(int)

In [271]:
# concatena e agrega
team_rows = pd.concat([home, away], ignore_index=True)

In [272]:
base_table = (
    team_rows
    .groupby(["nm_player", "nm_grpo", "team"], as_index=False)
    .agg(
        pts=("pts", "sum"),
        jogos=("team", "size"),
        v=("v", "sum"),
        e=("e", "sum"),
        d=("d", "sum"),
        gp=("gf", "sum"),
        gc=("ga", "sum"),
    )
)

base_table["sg"] = base_table["gp"] - base_table["gc"]

base_table

,nm_player,nm_grpo,team,pts,jogos,v,e,d,gp,gc,sg
0,ana nath,A,Coreia do Sul,1,3,0,1,2,3,7,-4
1,ana nath,A,Europa D,7,3,2,1,0,7,2,5
2,ana nath,A,México,6,3,2,0,1,9,5,4
3,ana nath,A,África do Sul,3,3,1,0,2,4,9,-5
4,ana nath,B,Canadá,7,3,2,1,0,7,3,4
...,...,...,...,...,...,...,...,...,...,...,...
187,washington,K,Uzbequistão,7,3,2,1,0,7,2,5
188,washington,L,Croácia,3,3,1,0,2,4,5,-1
189,washington,L,Gana,5,3,1,2,0,7,4,3
190,washington,L,Inglaterra,1,3,0,1,2,1,5,-4


In [273]:
base_table['rk'] = base_table.groupby(['nm_player','nm_grpo'])['pts'].rank(method='max', ascending=False).astype(int)


base_table[base_table['nm_player'] == 'ferolifil'].sort_values(['nm_grpo','rk'], ascending=[True, True])

,nm_player,nm_grpo,team,pts,jogos,v,e,d,gp,gc,sg,rk
48,ferolifil,A,Coreia do Sul,3,3,0,3,0,6,6,0,4
49,ferolifil,A,Europa D,3,3,0,3,0,3,3,0,4
50,ferolifil,A,México,3,3,0,3,0,1,1,0,4
51,ferolifil,A,África do Sul,3,3,0,3,0,6,6,0,4
52,ferolifil,B,Canadá,6,3,2,0,1,3,4,-1,2
55,ferolifil,B,Suíça,6,3,2,0,1,4,2,2,2
53,ferolifil,B,Catar,3,3,1,0,2,3,4,-1,4
54,ferolifil,B,Europa A,3,3,1,0,2,5,5,0,4
56,ferolifil,C,Brasil,9,3,3,0,0,7,1,6,1
57,ferolifil,C,Escócia,2,3,0,2,1,2,4,-2,4


## Critérios de Desempate

A classificação de seleções em cada grupo será determinada pelos pontos obtidos em todas as partidas do grupos. Se duas ou mais seleções empatarem em pontos, os critérios a seguir serão usados para determinar a classificação:

a. Mais pontos obtidos na partida de grupo jogada entre as seleções em questão;\
b. Maior saldo de gols na partida de grupo jogada entre as seleções em questão;\
c. Mais gols marcados na partida de grupo jogada entre as seleções em questão;

Se, depois de aplicados os critérios de A a C, seleções ainda estiverem empatadas, estes critérios serão aplicados novamente exclusivamente às partidas entre as seleções que ainda estão empatadas para determinar sua classificação final. Se este procedimento não levar a uma decisão, os critérios de D a H se aplicam.

d. Maior saldo de gols em todas as partidas do grupo;\
e. Mais gols marcados em todas as partidas do grupo;\
f. Melhor conduta ("fair play") em todas as partidas do grupo (apenas uma dedução pode ser aplicada a um jogador ou a um membro da comissão técnica/dirigente por partida):
- Cartão amarelo: −1 ponto;
- Cartão vermelho indireto (segundo cartão amarelo): −3 pontos;
- Cartão vermelho direto: −4 pontos;
- Cartão amarelo e cartão vermelho direto: −5 pontos;

g. Melhor posição no Ranking Mundial da FIFA mais recente;\
h. Melhor posição em Rankings Mundiais da FIFA mais antigos progressivamente até que as seleções sejam separadas;

In [274]:
# pontos por confronto direto
h2h = (
    team_rows
    .groupby(["nm_player", "nm_grpo", "team", "opp", "gf", "ga"], as_index=False)["pts"]
    .sum()
)

h2h[(h2h['nm_player'] == 'ferolifil') & (h2h['nm_grpo'] == 'A')].sort_values(['nm_grpo'], ascending=[True])

,nm_player,nm_grpo,team,opp,gf,ga,pts
144,ferolifil,A,Coreia do Sul,Europa D,1,1,1
145,ferolifil,A,Coreia do Sul,México,1,1,1
146,ferolifil,A,Coreia do Sul,África do Sul,4,4,1
147,ferolifil,A,Europa D,Coreia do Sul,1,1,1
148,ferolifil,A,Europa D,México,0,0,1
149,ferolifil,A,Europa D,África do Sul,2,2,1
150,ferolifil,A,México,Coreia do Sul,1,1,1
151,ferolifil,A,México,Europa D,0,0,1
152,ferolifil,A,México,África do Sul,0,0,1
153,ferolifil,A,África do Sul,Coreia do Sul,4,4,1


In [275]:
tot = base_table[["nm_player", "nm_grpo", "team", "pts"]].rename(columns={"pts": "total_pts"})

tot[(tot['nm_player'] == 'ferolifil') & (tot['nm_grpo'] == 'A')].sort_values(['nm_grpo'], ascending=[True])

,nm_player,nm_grpo,team,total_pts
48,ferolifil,A,Coreia do Sul,3
49,ferolifil,A,Europa D,3
50,ferolifil,A,México,3
51,ferolifil,A,África do Sul,3


In [276]:
h2h_2 = h2h.merge(tot, on=["nm_player", "nm_grpo", "team"])

h2h_2[(h2h_2['nm_player'] == 'ferolifil') & (h2h_2['nm_grpo'] == 'A')].sort_values(['nm_grpo'], ascending=[True])

,nm_player,nm_grpo,team,opp,gf,ga,pts,total_pts
144,ferolifil,A,Coreia do Sul,Europa D,1,1,1,3
145,ferolifil,A,Coreia do Sul,México,1,1,1,3
146,ferolifil,A,Coreia do Sul,África do Sul,4,4,1,3
147,ferolifil,A,Europa D,Coreia do Sul,1,1,1,3
148,ferolifil,A,Europa D,México,0,0,1,3
149,ferolifil,A,Europa D,África do Sul,2,2,1,3
150,ferolifil,A,México,Coreia do Sul,1,1,1,3
151,ferolifil,A,México,Europa D,0,0,1,3
152,ferolifil,A,México,África do Sul,0,0,1,3
153,ferolifil,A,África do Sul,Coreia do Sul,4,4,1,3


In [277]:
h2h_3 = h2h_2.merge(
    tot.rename(columns={"team": "opp", "total_pts": "opp_total_pts"}),
    on=["nm_player", "nm_grpo", "opp"]
)

h2h_3[(h2h_3['nm_player'] == 'ferolifil') & (h2h_3['nm_grpo'] == 'A')].sort_values(['nm_grpo'], ascending=[True])

,nm_player,nm_grpo,team,opp,gf,ga,pts,total_pts,opp_total_pts
144,ferolifil,A,Coreia do Sul,Europa D,1,1,1,3,3
145,ferolifil,A,Coreia do Sul,México,1,1,1,3,3
146,ferolifil,A,Coreia do Sul,África do Sul,4,4,1,3,3
147,ferolifil,A,Europa D,Coreia do Sul,1,1,1,3,3
148,ferolifil,A,Europa D,México,0,0,1,3,3
149,ferolifil,A,Europa D,África do Sul,2,2,1,3,3
150,ferolifil,A,México,Coreia do Sul,1,1,1,3,3
151,ferolifil,A,México,Europa D,0,0,1,3,3
152,ferolifil,A,México,África do Sul,0,0,1,3,3
153,ferolifil,A,África do Sul,Coreia do Sul,4,4,1,3,3


In [278]:
# mantém só jogos entre equipes com o mesmo total de pontos
h2h_tied = h2h_3[h2h_3["total_pts"] == h2h_3["opp_total_pts"]]

h2h_tied[h2h_tied['nm_player'] == 'ferolifil'].sort_values(['nm_grpo'], ascending=[True])

,nm_player,nm_grpo,team,opp,gf,ga,pts,total_pts,opp_total_pts
144,ferolifil,A,Coreia do Sul,Europa D,1,1,1,3,3
145,ferolifil,A,Coreia do Sul,México,1,1,1,3,3
146,ferolifil,A,Coreia do Sul,África do Sul,4,4,1,3,3
147,ferolifil,A,Europa D,Coreia do Sul,1,1,1,3,3
148,ferolifil,A,Europa D,México,0,0,1,3,3
149,ferolifil,A,Europa D,África do Sul,2,2,1,3,3
150,ferolifil,A,México,Coreia do Sul,1,1,1,3,3
151,ferolifil,A,México,Europa D,0,0,1,3,3
152,ferolifil,A,México,África do Sul,0,0,1,3,3
153,ferolifil,A,África do Sul,Coreia do Sul,4,4,1,3,3


In [279]:
h2h_pts = (
    h2h_tied
    .groupby(["nm_player", "nm_grpo", "team"], as_index=False)[["pts", "gf", "ga"]]
    .sum()
    .rename(columns={"pts": "h2h_pts", "gf": "h2h_gf", "ga": "h2h_ga"})
)

h2h_pts["h2h_sg"] = h2h_pts["h2h_gf"] - h2h_pts["h2h_ga"]

h2h_pts[(h2h_pts['nm_player'] == 'ferolifil') & (h2h_pts['nm_grpo'] == 'A')].sort_values(['nm_grpo'], ascending=[True])



,nm_player,nm_grpo,team,h2h_pts,h2h_gf,h2h_ga,h2h_sg
14,ferolifil,A,Coreia do Sul,3,6,6,0
15,ferolifil,A,Europa D,3,3,3,0
16,ferolifil,A,México,3,1,1,0
17,ferolifil,A,África do Sul,3,6,6,0


In [280]:
h2h_pts = h2h_pts.assign(
    tie_key=list(zip(h2h_pts.h2h_pts, h2h_pts.h2h_sg, h2h_pts.h2h_gf))
)

h2h_pts["rk2"] = (
    h2h_pts
    .groupby(["nm_player", "nm_grpo"])["tie_key"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

h2h_pts = h2h_pts.drop(columns=["tie_key", "h2h_sg", "h2h_gf", "h2h_ga", "h2h_pts"])


In [281]:
base_table[base_table['nm_player'] == 'ferolifil'].sort_values(['nm_grpo','rk'], ascending=[True, True])

,nm_player,nm_grpo,team,pts,jogos,v,e,d,gp,gc,sg,rk
48,ferolifil,A,Coreia do Sul,3,3,0,3,0,6,6,0,4
49,ferolifil,A,Europa D,3,3,0,3,0,3,3,0,4
50,ferolifil,A,México,3,3,0,3,0,1,1,0,4
51,ferolifil,A,África do Sul,3,3,0,3,0,6,6,0,4
52,ferolifil,B,Canadá,6,3,2,0,1,3,4,-1,2
55,ferolifil,B,Suíça,6,3,2,0,1,4,2,2,2
53,ferolifil,B,Catar,3,3,1,0,2,3,4,-1,4
54,ferolifil,B,Europa A,3,3,1,0,2,5,5,0,4
56,ferolifil,C,Brasil,9,3,3,0,0,7,1,6,1
57,ferolifil,C,Escócia,2,3,0,2,1,2,4,-2,4


In [282]:
h2h_pts[h2h_pts['nm_player'] == 'ferolifil'].sort_values(['nm_grpo','rk2'], ascending=[True, True])

,nm_player,nm_grpo,team,rk2
14,ferolifil,A,Coreia do Sul,1
17,ferolifil,A,África do Sul,1
15,ferolifil,A,Europa D,2
16,ferolifil,A,México,3
21,ferolifil,B,Suíça,1
19,ferolifil,B,Catar,2
20,ferolifil,B,Europa A,3
18,ferolifil,B,Canadá,4
24,ferolifil,C,Marrocos,1
22,ferolifil,C,Escócia,2


In [283]:
base_table_2 = base_table.merge(h2h_pts, on=["nm_player", "nm_grpo", "team"], how="left").fillna(0)

base_table_2[base_table_2['nm_player'] == 'ferolifil'].sort_values(['nm_grpo','rk'], ascending=[True, True])

,nm_player,nm_grpo,team,pts,jogos,v,e,d,gp,gc,sg,rk,rk2
48,ferolifil,A,Coreia do Sul,3,3,0,3,0,6,6,0,4,1.0
49,ferolifil,A,Europa D,3,3,0,3,0,3,3,0,4,2.0
50,ferolifil,A,México,3,3,0,3,0,1,1,0,4,3.0
51,ferolifil,A,África do Sul,3,3,0,3,0,6,6,0,4,1.0
52,ferolifil,B,Canadá,6,3,2,0,1,3,4,-1,2,4.0
55,ferolifil,B,Suíça,6,3,2,0,1,4,2,2,2,1.0
53,ferolifil,B,Catar,3,3,1,0,2,3,4,-1,4,2.0
54,ferolifil,B,Europa A,3,3,1,0,2,5,5,0,4,3.0
56,ferolifil,C,Brasil,9,3,3,0,0,7,1,6,1,0.0
57,ferolifil,C,Escócia,2,3,0,2,1,2,4,-2,4,2.0


In [284]:
base_table_3 = (
    base_table_2
        .merge(
            df_paises.drop(columns=["id_pais", "nm_grpo"]).rename(columns={"nm_pais": "team"}), 
            on="team", 
            how="left"
        )
    )

In [285]:
base_table_3 = base_table_3.assign(
    tie_key=list(zip(
        base_table_3["rk"],
        base_table_3["rk2"],
        base_table_3["sg"],
        base_table_3["gp"],
        base_table_3["nm_pais_ajst"],
    ))
)

base_table_3["pos"] = (
    base_table_3
    .groupby(["nm_player", "nm_grpo"])["tie_key"]
    .rank(method="dense", ascending=True)
    .astype(int)
)

In [286]:
base_table_3 = base_table_3.drop(columns=["tie_key", "rk", "rk2", "nm_pais_ajst"])

base_table_3[base_table_3['nm_player'] == 'ferolifil'].sort_values(['nm_grpo','pos'], ascending=[True, True])


,nm_player,nm_grpo,team,pts,jogos,v,e,d,gp,gc,sg,pos
51,ferolifil,A,África do Sul,3,3,0,3,0,6,6,0,1
48,ferolifil,A,Coreia do Sul,3,3,0,3,0,6,6,0,2
49,ferolifil,A,Europa D,3,3,0,3,0,3,3,0,3
50,ferolifil,A,México,3,3,0,3,0,1,1,0,4
55,ferolifil,B,Suíça,6,3,2,0,1,4,2,2,1
52,ferolifil,B,Canadá,6,3,2,0,1,3,4,-1,2
53,ferolifil,B,Catar,3,3,1,0,2,3,4,-1,3
54,ferolifil,B,Europa A,3,3,1,0,2,5,5,0,4
56,ferolifil,C,Brasil,9,3,3,0,0,7,1,6,1
59,ferolifil,C,Marrocos,2,3,0,2,1,2,5,-3,2


In [287]:
team_rows[(team_rows['nm_player'] == 'ferolifil') & (team_rows['team'] == "Catar")].sort_values(['nm_grpo', 'team'], ascending=[True, True])

,nm_player,nm_cfr,team,gf,opp,ga,result_ref_casa,id_pais,nm_pais_ajst,nm_grpo,pts,v,e,d
86,ferolifil,Catar x Suíça,Catar,0,Suíça,1,D,34,CATAR,B,0,0,0,1
372,ferolifil,Canadá x Catar,Catar,0,Canadá,1,V,24,CANADA,B,0,0,0,1
390,ferolifil,Europa A x Catar,Catar,3,Europa A,2,D,43,EUROPA A,B,3,1,0,0
